In [1]:
import torch
import pandas as pd
import numpy as np
from transformers import AutoTokenizer, AutoModel
from tqdm import tqdm

site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [3]:
train_csv = "../../general_input/train/train_data_prepared.csv"
embeddings_path = "../input/train_embeddings.npy"

In [4]:
model_name = "facebook/esm2_t33_650M_UR50D"
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [5]:
model = AutoModel.from_pretrained(model_name, torch_dtype=torch.float16).to(device)
model.eval()

Some weights of EsmModel were not initialized from the model checkpoint at facebook/esm2_t33_650M_UR50D and are newly initialized: ['esm.pooler.dense.bias', 'esm.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


EsmModel(
  (embeddings): EsmEmbeddings(
    (word_embeddings): Embedding(33, 1280, padding_idx=1)
    (dropout): Dropout(p=0.0, inplace=False)
    (position_embeddings): Embedding(1026, 1280, padding_idx=1)
  )
  (encoder): EsmEncoder(
    (layer): ModuleList(
      (0-32): 33 x EsmLayer(
        (attention): EsmAttention(
          (self): EsmSelfAttention(
            (query): Linear(in_features=1280, out_features=1280, bias=True)
            (key): Linear(in_features=1280, out_features=1280, bias=True)
            (value): Linear(in_features=1280, out_features=1280, bias=True)
            (dropout): Dropout(p=0.0, inplace=False)
            (rotary_embeddings): RotaryEmbedding()
          )
          (output): EsmSelfOutput(
            (dense): Linear(in_features=1280, out_features=1280, bias=True)
            (dropout): Dropout(p=0.0, inplace=False)
          )
          (LayerNorm): LayerNorm((1280,), eps=1e-05, elementwise_affine=True)
        )
        (intermediate): EsmInter

In [6]:
df = pd.read_csv(train_csv)
sequences = df['sequence'].tolist()

In [7]:
def get_embeddings_low_vram(sequences, batch_size=2):
    embeddings = []
    
    for i in tqdm(range(0, len(sequences), batch_size)):
        batch_seqs = sequences[i : i + batch_size]
        
        inputs = tokenizer(batch_seqs, return_tensors="pt", padding=True, truncation=True, max_length=1024)
        inputs = {k: v.to(device) for k, v in inputs.items()}
        
        with torch.no_grad():
            outputs = model(**inputs)
            
        batch_embeddings = outputs.last_hidden_state.mean(dim=1).float()
        
        embeddings.append(batch_embeddings.cpu().numpy())
        
    return np.concatenate(embeddings, axis=0)

In [8]:
try:
    embeddings = get_embeddings_low_vram(sequences, batch_size=2)
    print(f"Embeddings generated. Shape: {embeddings.shape}")
    np.save(embeddings_path, embeddings)
    print("Saved embeddings to disk.")
except RuntimeError as e:
    if "out of memory" in str(e):
        print("MemoryError: Not enough GPU memory to process the batch size. Try reducing the batch size.")
    else:
        print(f"An unexpected error ocurred: {e}")

  5%|▍         | 1882/41202 [17:50<6:12:44,  1.76it/s] 


KeyboardInterrupt: 